# Notebook 05: Model Serving

In this notebook, you will learn how to **serve** your trained model so that other systems (and people) can actually use it. We will build a REST API with FastAPI, test it interactively, send batch predictions, and handle errors gracefully.

### What You Will Learn

- The two main serving patterns: real-time and batch
- How REST APIs work (HTTP, JSON, status codes)
- How to define request/response schemas with Pydantic
- How to test an API without running a server
- How to send batch predictions and measure throughput
- How the API validates input and returns useful errors
- How to run the server for real (locally and with Docker)

### Prerequisites

- Notebooks 00 through 04 completed
- `pip install -e '.[dev,test]'` from the project root

---
## 1. Why Serving Matters

**Your model is useless if nobody can use it.**

Think about it: you spent days collecting data, engineering features, training models, and tuning hyperparameters. You have a beautiful `model.predict(X)` call that works perfectly in your notebook. But your notebook is not a product. The operations team cannot call your Jupyter cell. The mobile app cannot import your pandas DataFrame. The dashboard cannot read your pickle file.

To make your model useful, you need to **serve** it -- wrap it in an interface that other systems can call.

### Two Serving Patterns

There are two fundamental patterns for serving predictions:

| Pattern | How It Works | Latency | Use Case |
|---------|-------------|---------|----------|
| **Real-time (online)** | Client sends a request, gets a prediction back immediately via a REST API | Milliseconds | A building management system asks: "What will demand be in the next hour?" and needs an answer *now* to adjust HVAC settings. |
| **Batch (offline)** | A scheduled job processes thousands of predictions at once and writes results to a database or file | Minutes to hours | Every night at midnight, generate tomorrow's 24-hour forecast for all 500 buildings and store the results for the operations dashboard. |

### When to Use Which?

- **Use real-time** when the consumer needs an answer immediately and the input is not known in advance. Examples: fraud detection at transaction time, autocomplete suggestions, real-time energy adjustments.
- **Use batch** when you can predict in advance, when latency does not matter, or when you need to process a large volume efficiently. Examples: next-day demand forecasts, weekly reports, populating a cache.

In practice, most production systems use **both**. Our energy forecasting system exposes a real-time API for on-demand predictions *and* supports batch endpoints for bulk processing.

In this notebook, we will build and test both patterns.

---
## 2. REST API Primer

If you have never built an API before, this section is for you. If you already know what REST is, feel free to skim.

### What is an API?

An API (Application Programming Interface) is a contract between two programs. One program (the **client**) sends a request, and the other program (the **server**) processes it and sends back a response.

```
Client (mobile app, dashboard, another service)
   |
   |  HTTP Request: "Predict energy demand for building B001 at 2pm"
   v
Server (our FastAPI application, with the ML model loaded)
   |
   |  HTTP Response: {"predicted_kwh": 245.7, "confidence_lower": 221.1, ...}
   v
Client receives the prediction and uses it
```

### HTTP Methods

HTTP defines several methods (verbs) for different operations. We use two:

- **GET** -- Retrieve information. No data sent in the body. Example: `GET /health` returns the server's health status.
- **POST** -- Send data and get a result. Data goes in the request body as JSON. Example: `POST /predict` with a JSON body containing temperature, humidity, etc.

### JSON

JSON (JavaScript Object Notation) is the standard data format for APIs. It looks like a Python dictionary:

```json
{
  "building_id": "B001",
  "temperature": 25.0,
  "humidity": 60.0
}
```

### Status Codes

Every HTTP response includes a numeric status code that tells the client what happened:

| Code | Meaning | When You See It |
|------|---------|----------------|
| **200** | OK | Everything worked. Here is your prediction. |
| **422** | Unprocessable Entity | Your request was malformed -- missing a field, wrong data type, humidity of 150%. |
| **500** | Internal Server Error | Something broke on the server side -- a bug, the model failed, etc. |
| **503** | Service Unavailable | The model is not loaded yet. Try again in a moment. |

### Why FastAPI?

FastAPI is a modern Python web framework that makes building APIs easy:

- **Automatic validation** -- Define your data schema with Pydantic, and FastAPI validates every request for free. If someone sends `humidity: "not a number"`, the client gets a clear 422 error without you writing a single `if` statement.
- **Automatic documentation** -- FastAPI generates interactive API docs (Swagger UI) at `/docs`. You can test your API from a browser.
- **Fast** -- Built on Starlette and uvicorn, it handles thousands of requests per second.
- **Python-native** -- Type hints are the API contract. If you can write a Python function with type annotations, you can build an API.

---
## 3. Request/Response Schemas

Before we write any API logic, we define the **shape** of the data going in and out. This is done with Pydantic models -- Python classes that describe and validate data.

Our project defines these schemas in `src/energy_forecast/serving/schemas.py`. Let's look at them.

In [ ]:
import sys
sys.path.insert(0, '../src')

from energy_forecast.serving.schemas import (
    PredictionRequest,
    PredictionResponse,
    BatchPredictionRequest,
    BatchPredictionResponse,
    HealthResponse,
    ErrorResponse,
)

# Let's inspect what fields each schema expects
print("=== PredictionRequest ===")
for name, field in PredictionRequest.model_fields.items():
    print(f"  {name}: {field.annotation} -- {field.description}")

print("\n=== PredictionResponse ===")
for name, field in PredictionResponse.model_fields.items():
    print(f"  {name}: {field.annotation}")

print("\n=== HealthResponse ===")
for name, field in HealthResponse.model_fields.items():
    print(f"  {name}: {field.annotation}")

In [ ]:
# Create a sample request and see the JSON that would be sent to the API
sample = PredictionRequest(
    timestamp="2023-06-15T14:00:00",
    building_id="B001",
    temperature=25.0,
    humidity=60.0,
)

print("Sample PredictionRequest as JSON:")
print(sample.model_dump_json(indent=2))

In [ ]:
# Pydantic validates data automatically.
# Let's see what happens with invalid humidity (must be 0-100):
from pydantic import ValidationError

try:
    bad_request = PredictionRequest(
        timestamp="2023-06-15T14:00:00",
        building_id="B001",
        temperature=25.0,
        humidity=150.0,  # Invalid! Max is 100.
    )
except ValidationError as e:
    print("Validation error caught!")
    print(e)

Notice that we never wrote any `if humidity > 100` checks. The Pydantic model has `ge=0.0, le=100.0` constraints on the `humidity` field, and validation happens automatically. This is one of the great advantages of schema-driven APIs: **the contract is enforced by the framework**, not by hand-written code scattered throughout your application.

---
## 4. Testing the API

Here is a powerful trick: **you do not need to start a server to test your API.** FastAPI provides a `TestClient` that simulates HTTP requests in-process. This is how you write automated tests for your API, and it is also perfect for interactive exploration in a notebook.

Under the hood, `TestClient` uses the same request/response cycle as a real server, including all middleware and validation. The only difference is that no network is involved.

In [ ]:
from fastapi.testclient import TestClient
from energy_forecast.serving.app import create_app

# Create the FastAPI application and a test client
app = create_app()
client = TestClient(app)

# --- Health Check ---
response = client.get("/health")
print(f"GET /health")
print(f"  Status code: {response.status_code}")
print(f"  Response: {response.json()}")

In [ ]:
# --- Single Prediction ---
payload = {
    "timestamp": "2023-06-15T14:00:00",
    "building_id": "B001",
    "temperature": 25.0,
    "humidity": 60.0,
}

response = client.post("/predict", json=payload)
print(f"POST /predict")
print(f"  Status code: {response.status_code}")
print(f"  Prediction: {response.json()}")

In [ ]:
# --- Model Info ---
response = client.get("/model/info")
print(f"GET /model/info")
print(f"  Status code: {response.status_code}")

import json
print(f"  Response:\n{json.dumps(response.json(), indent=2)}")

Notice the flow: we created the app, wrapped it in a `TestClient`, and then used `.get()` and `.post()` to simulate real HTTP calls. The status codes, JSON responses, and validation all behave exactly as they would with a real running server.

---
## 5. Batch Predictions

Real-time is great for one-off requests, but what if you need predictions for the next 24 hours? Or for 500 buildings? That is where batch predictions come in.

Our API has a `/predict/batch` endpoint that accepts a list of prediction requests and returns all results at once, along with the total processing time.

In [ ]:
import time
from datetime import datetime, timedelta

# Generate 24 prediction requests -- one for each hour of the day
base_time = datetime(2023, 6, 15, 0, 0, 0)
hourly_requests = []

for hour in range(24):
    ts = base_time + timedelta(hours=hour)
    # Simulate realistic temperature: cooler at night, warmer midday
    temp = 18.0 + 8.0 * abs(12 - hour) / 12.0 * (-1 if hour < 6 or hour > 18 else 1)
    hourly_requests.append({
        "timestamp": ts.isoformat(),
        "building_id": "B001",
        "temperature": round(temp, 1),
        "humidity": 55.0 + hour * 0.5,
    })

print(f"Prepared {len(hourly_requests)} prediction requests (one per hour).")
print(f"First: {hourly_requests[0]['timestamp']}")
print(f"Last:  {hourly_requests[-1]['timestamp']}")

# Send the batch request
start = time.perf_counter()
response = client.post("/predict/batch", json={"predictions": hourly_requests})
elapsed = (time.perf_counter() - start) * 1000

print(f"\nBatch POST /predict/batch")
print(f"  Status code: {response.status_code}")

data = response.json()
print(f"  Predictions returned: {len(data['predictions'])}")
print(f"  Server-reported processing time: {data['processing_time_ms']:.2f} ms")
print(f"  Total round-trip time: {elapsed:.2f} ms")

# Show a few results
print("\n  Hour | Predicted kWh | Confidence Interval")
print("  " + "-" * 50)
for pred in data['predictions'][:6]:
    ts = pred['timestamp']
    hour = ts.split('T')[1][:5]
    print(f"  {hour} | {pred['predicted_kwh']:>13.1f} | [{pred['confidence_lower']:.1f}, {pred['confidence_upper']:.1f}]")
print(f"  ...  | ({len(data['predictions']) - 6} more rows)")

---
## 6. Error Handling

A production API must handle bad input gracefully. It should **never crash**. Instead, it should return a clear error message with an appropriate HTTP status code so the client knows what went wrong and how to fix it.

Let's send some intentionally bad requests and see how the API responds.

In [ ]:
# --- Test 1: Missing required fields ---
print("Test 1: Missing required fields")
response = client.post("/predict", json={
    "building_id": "B001",
    # Missing: timestamp, temperature, humidity
})
print(f"  Status: {response.status_code}")
print(f"  Error:  {json.dumps(response.json(), indent=4)}")

print()

# --- Test 2: Invalid humidity (out of range) ---
print("Test 2: Humidity out of range (must be 0-100)")
response = client.post("/predict", json={
    "timestamp": "2023-06-15T14:00:00",
    "building_id": "B001",
    "temperature": 25.0,
    "humidity": -10.0,  # Negative humidity is impossible
})
print(f"  Status: {response.status_code}")
print(f"  Error:  {json.dumps(response.json(), indent=4)}")

print()

# --- Test 3: Wrong data type ---
print("Test 3: Wrong data type (string where number expected)")
response = client.post("/predict", json={
    "timestamp": "2023-06-15T14:00:00",
    "building_id": "B001",
    "temperature": "not_a_number",
    "humidity": 60.0,
})
print(f"  Status: {response.status_code}")
print(f"  Error:  {json.dumps(response.json(), indent=4)}")

print()

# --- Test 4: Invalid timestamp format ---
print("Test 4: Invalid timestamp format")
response = client.post("/predict", json={
    "timestamp": "June 15th at 2pm",
    "building_id": "B001",
    "temperature": 25.0,
    "humidity": 60.0,
})
print(f"  Status: {response.status_code}")
print(f"  Error:  {json.dumps(response.json(), indent=4)}")

In [ ]:
# --- Test 5: Empty batch ---
print("Test 5: Empty batch (min_length=1 required)")
response = client.post("/predict/batch", json={
    "predictions": []
})
print(f"  Status: {response.status_code}")
print(f"  Error:  {json.dumps(response.json(), indent=4)}")

print()

# --- Test 6: Completely empty body ---
print("Test 6: No JSON body at all")
response = client.post("/predict")
print(f"  Status: {response.status_code}")
print(f"  Error:  {json.dumps(response.json(), indent=4)}")

Notice the pattern: every error returns a meaningful status code and a JSON body describing what went wrong. The client never sees a stack trace or a raw Python exception. This is how production APIs should behave.

FastAPI + Pydantic give us most of this for free. The `ge=0.0, le=100.0` constraint on humidity, the `min_length=1` on the batch list, and the type annotations all generate automatic validation. We only need custom error handlers for business logic errors (like the model not being loaded).

---
## 7. Running the Server for Real

So far we have tested our API using `TestClient`, which is great for development and automated tests. But at some point you need an actual running server that accepts requests over the network.

### Option 1: Make Command

From the project root:

```bash
make serve
```

This starts the FastAPI server with uvicorn on `http://localhost:8000`.

### Option 2: Direct Uvicorn

```bash
uvicorn energy_forecast.serving.app:create_app --factory --host 0.0.0.0 --port 8000 --reload
```

The `--reload` flag watches for code changes and restarts automatically (great for development).

### Option 3: Docker Compose

For a production-like setup with all services (API + Prometheus + Grafana):

```bash
docker compose up -d api
```

### Interactive Documentation (Swagger UI)

Once the server is running, open your browser to:

- **Swagger UI**: `http://localhost:8000/docs` -- Interactive API docs where you can try every endpoint directly from the browser. Click "Try it out", fill in the fields, and hit "Execute".
- **ReDoc**: `http://localhost:8000/redoc` -- A cleaner, read-only version of the docs.

Both are generated automatically from our Pydantic schemas and route definitions. You never need to write API documentation by hand.

### Testing with curl

From the command line:

```bash
# Health check
curl http://localhost:8000/health

# Single prediction
curl -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"timestamp": "2023-06-15T14:00:00", "building_id": "B001", "temperature": 25.0, "humidity": 60.0}'
```

---
## 8. Exercises

### Exercise 1: Multi-Building Batch

Write code that sends a batch prediction request for **3 different buildings** (`B001`, `B002`, `B003`), each with **8 hours** of predictions (08:00 to 15:00). That is 24 total predictions. Print the results grouped by building.

*Hint: Build a list of 24 request dicts and POST them to `/predict/batch`.*

### Exercise 2: Response Time Benchmark

Measure how long the API takes to respond to increasing batch sizes: 1, 10, 50, 100, and 500 predictions. Plot batch size vs. response time. Is the relationship linear? What does this tell you about the overhead of each request?

*Hint: Use `time.perf_counter()` around each `client.post()` call. Use matplotlib to plot the results.*

---
## 9. Key Takeaways

1. **Serving is how your model reaches users.** A model that cannot be called by other systems is a science experiment, not a product.

2. **Real-time and batch are complementary patterns.** Real-time for on-demand predictions (milliseconds), batch for bulk processing (scheduled). Most production systems need both.

3. **Schema-driven APIs are self-validating.** Pydantic models define the contract, FastAPI enforces it, and invalid input gets a clear 422 error without any manual `if` checks.

4. **TestClient lets you test without a server.** Write automated tests and explore interactively in notebooks. Same behavior as a real server, zero setup.

5. **Error handling is a feature, not an afterthought.** A production API must never crash. Every failure should return a meaningful status code and message.

6. **FastAPI generates documentation automatically.** Swagger UI at `/docs` gives you interactive API exploration for free.

### What's Next

Your model is now accessible to the world. But how do you know it is working well? In **Notebook 06: Monitoring and Drift Detection**, we will build the observability layer that watches your model in production and alerts you when things start to go wrong.